# MapReduce

Aluno: Kaike Armond Costa \
Curso: Ciência de Dados e Inteligência Artificial \
Professor: Alexandre Vaz Roriz

Este notebook traz a programação de MapReduce (Map → Shuffle/Sort → Reduce) para responder a seis perguntas sobre uma amostra de 1 milhão de viagens de táxi amarelo de Nova York, realizadas ao longo de 6 meses de 2024.


## Antes de começar: carregando o arquivo de dados no Colab

Este notebook foi feito para rodar no **Google Colab**. Como não podemos usar nenhuma biblioteca, o CSV precisa ser enviado manualmente para o ambiente do Colab:

1. No menu lateral esquerdo do Colab, clique no ícone de pasta Arquivos
2. Clique no ícone de upload e selecione o arquivo `nyc_tripdata_2024_sample_1M.csv` no seu computador.
3. Aguarde o upload terminar.
4. Confirme que o arquivo aparece como `/content/nyc_tripdata_2024_sample_1M.csv` na aba de arquivos.


In [27]:
# Caminho do arquivo CSV no ambiente do Colab.
CAMINHO_ARQUIVO = "/content/nyc_tripdata_2024_sample_1M.csv"


### Funções de MapReduce

In [28]:
def mapear(funcao, dados):
    # roda a função de map em cada linha e junta tudo numa lista só de (chave, valor)
    saida = []
    for item in dados:
        saida += funcao(item)
    return saida


def agrupar(pares):
    # fase de shuffle: agrupa os valores pela chave
    grupos = {}
    for chave, valor in pares:
        if chave not in grupos:
            grupos[chave] = []
        grupos[chave].append(valor)
    return grupos


def reduzir(funcao, grupos):
    # aplica a função de reduce em cada grupo
    saida = {}
    for chave, valores in grupos.items():
        saida[chave] = funcao(chave, valores)
    return saida


def map_reduce(dados, funcao_map, funcao_reduce):
    pares = mapear(funcao_map, dados)
    grupos = agrupar(pares)
    return reduzir(funcao_reduce, grupos)


### Carregar o CSV

In [29]:
def carregar_dados(caminho):
    # Carregamento de dados do CSV
    # Faço cada linha virar uma tupla: (data_hora_embarque, distancia, pagamento, tarifa, total)
    dados = []
    arquivo = open(caminho, "r", encoding="utf-8")

    cabecalho = arquivo.readline().strip().split(",")
    idx_data = cabecalho.index("tpep_pickup_datetime")
    idx_dist = cabecalho.index("trip_distance")
    idx_pag = cabecalho.index("payment_type")
    idx_tarifa = cabecalho.index("fare_amount")
    idx_total = cabecalho.index("total_amount")

    for linha in arquivo:
        c = linha.strip().split(",")
        dados.append((
            c[idx_data],
            float(c[idx_dist]),
            int(float(c[idx_pag])),
            float(c[idx_tarifa]),
            float(c[idx_total]),
        ))

    arquivo.close()
    return dados
'''
  Confiro a quantidade de linhas carregadas, a lista dados agora contém uma
  lista de tuplas, onde cada tupla representa uma linha do CSV.
'''

viagens = carregar_dados(caminho)
len(viagens)


1000000

## Dicionário de tipos de pagamento

Segundo o dicionário de dados oficial da NYC TLC, a coluna `payment_type` usa os seguintes códigos:

| Código | Significado |
|---|---|
| 0 | Flex Fare trip |
| 1 | Credit card |
| 2 | Cash |
| 3 | No charge |
| 4 | Dispute |
| 5 | Unknown |
| 6 | Voided trip |

Usamos esse mapeamento apenas para exibir os resultados das Questões 1 e 2 de forma legível.

In [30]:
# Defino o tipo de pagamento invez de usar o codigo específico.
nomes_pagamento = {0: "Flex Fare", 1: "Credit card", 2: "Cash", 3: "No charge", 4: "Dispute", 5: "Unknown", 6: "Voided trip"}


### 1) Número de viagens por tipo de pagamento
Map: cada viagem gera o par (tipo_pagamento, 1), logo "1 viagem" para aquele tipo de pagamento. Reduce: somamos todos os 1s de cada chave, obtendo a contagem total de viagens por tipo de pagamento.

In [31]:
# v[2] refere-se ao terceiro elemento da tupla: payment_type(Tipo de pagamento)
def map_q1(v):
    return [(v[2], 1)]

# Realizo a soma de todas as viagens para aquela chave: payment_type
def reduce_q1(chave, valores):
    return sum(valores)

# Exibo o tipo de pagamento e a contagem total de viagens associada a ele
r1 = map_reduce(viagens, map_q1, reduce_q1)
for k in sorted(r1):
    print(nomes_pagamento.get(k, k),":", r1[k])


Flex Fare : 97124
Credit card : 743405
Cash : 136221
No charge : 6707
Dispute : 16543


### 2) Receita total por tipo de pagamento
**Map:** cada viagem gera o par `(tipo_pagamento, valor_total)`, usando a coluna `total_amount`
(o valor total efetivamente cobrado do passageiro — tarifa + taxas + pedágios etc.)\
**Reduce:** somamos os valores de cada chave, obtendo a receita total por tipo de pagamento.


In [32]:
'''
v[2] é o payment_type (tipo de pagamento), que será a chave para o agrupamento.
v[4] é o total_amount (valor total da viagem), que será o valor associado a essa chave.
Portanto, para cada viagem, esta função retorna um par
(tipo_de_pagamento, valor_total_da_viagem).
'''
def map_q2(v):
    return [(v[2], v[4])]

# Somo todos os valores de total_amount com a chave associada
# Para resultar na receita total por tipo de pagamento
def reduce_q2(chave, valores):
    return sum(valores)

# Exibo a receita total associado com o tipo de pagamento
r2 = map_reduce(viagens, map_q2, reduce_q2)
for k in sorted(r2):
    print(nomes_pagamento.get(k, k), "->", round(r2[k], 2))


Flex Fare -> 2376069.77
Credit card -> 21785219.95
Cash -> 3168095.9
No charge -> 53932.48
Dispute -> 25214.51


### 3) Tarifa média
Map: cada viagem gera o par ("todas_as_viagens", valor_tarifa), usando a coluna fare_amount (a tarifa calculada pelo taxímetro, sem taxas e sobretaxas). Reduce: soma de todos os valores dividida pela quantidade de valores = média.

In [33]:
# v[3] refere-se à fare_amount (valor da tarifa) da viagem.
def map_q3(v):
    return [("media", v[3])]

# Retorno a media das tarifas
def reduce_q3(chave, valores):
    return sum(valores) / len(valores)

# Exibo a tarifa média
r3 = map_reduce(viagens, map_q3, reduce_q3)
print(r3["media"])


18.86029663


### 4) Viagem mais longa (data/hora)

**Map:** cada viagem gera o par `("todas_as_viagens", (distancia_viagem, data_hora_embarque)) ` \
**Reduce:** para a lista de pares `(distância, data/hora)` de uma chave, usamos a função nativa `max()` para encontrar o par cuja distância é a maior.


In [34]:
# v[1] refere-se à trip_distance (distância da viagem)
# v[0] refere-se à tpep_pickup_datetime (data e hora de embarque)
def map_q4(v):
    return [("maior", (v[1], v[0]))]

# max(valores) para encontrar o valor maximo na lista de tuplas
def reduce_q4(chave, valores):
    return max(valores)

# Exibo a maior distancia de viagem e sua data/hora
r4 = map_reduce(viagens, map_q4, reduce_q4)
print(r4["maior"])


(86789.2, '2024-05-10 17:33:00')


### 5) Quantidade de viagens por hora
**Map:** cada viagem gera o par `(hora_do_dia, 1)`. \
**Reduce:** soma das ocorrências de cada hora.

In [35]:
# v[0] refere-se à tpep_pickup_datetime (data e hora de embarque)
# A parte [11:13] extrai os caracteres que representam a hora
def map_q5(v):
    hora = int(v[0][11:13])
    return [(hora, 1)]

# Somo a lista de valores
# O resultado é a contagem total de viagens para aquela chave (hora do dia)
def reduce_q5(chave, valores):
    return sum(valores)

# Exibo a quantidade de viagens por hora do dia
r5 = map_reduce(viagens, map_q5, reduce_q5)
for h in sorted(r5):
    print(h, "->", r5[h])


0 -> 29165
1 -> 18822
2 -> 12280
3 -> 8281
4 -> 6054
5 -> 6194
6 -> 13966
7 -> 28065
8 -> 38308
9 -> 42309
10 -> 44804
11 -> 48295
12 -> 53128
13 -> 55362
14 -> 59345
15 -> 60205
16 -> 61563
17 -> 67880
18 -> 71403
19 -> 62752
20 -> 56542
21 -> 58333
22 -> 54581
23 -> 42363


### 6) Distância total por hora
mesma ideia da 5, mas somando trip_distance

In [36]:
# v[0] refere-se à tpep_pickup_datetime (data e hora de embarque)
# Para cada viagem, esta função emite um par (hora, distância).
# v[1] (que representa a trip_distance ou distância da viagem) é o valor associado a essa chave.
def map_q6(v):
    hora = int(v[0][11:13])
    return [(hora, v[1])]

# Somo as distâncias da lista de valores
# Resultando na distância total para aquela chave (hora do dia)
def reduce_q6(chave, valores):
    return sum(valores)

# Exibo a Distância total por hora
r6 = map_reduce(viagens, map_q6, reduce_q6)
for h in sorted(r6):
    print(h, "-->", round(r6[h], 2))


0 --> 109568.73
1 --> 60695.19
2 --> 36643.43
3 --> 29745.9
4 --> 28350.3
5 --> 91136.53
6 --> 131056.86
7 --> 195237.07
8 --> 169111.73
9 --> 222652.02
10 --> 138342.76
11 --> 145189.3
12 --> 166186.16
13 --> 190847.83
14 --> 215732.27
15 --> 312374.38
16 --> 238718.29
17 --> 321659.21
18 --> 252601.19
19 --> 265021.98
20 --> 267461.66
21 --> 283775.08
22 --> 189960.49
23 --> 161753.45
